In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# Run this first, then restart runtime when prompted
# ─────────────────────────────────────────────────────────────

#!pip install vllm==0.3.3
#!pip install datasets transformers
#!pip install lightgbm scikit-learn pandas numpy matplotlib seaborn



In [ ]:
import vllm
print(vllm.__version__)

0.19.0


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Verify GPU
# ─────────────────────────────────────────────────────────────

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


Mon Apr 13 10:19:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Imports
# ─────────────────────────────────────────────────────────────

import os
import json
import time
import pandas as pd
import numpy as np
from datasets import load_dataset
from typing import List, Dict, Optional

print("All imports successful!")

All imports successful!


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Load ShareGPT Dataset
# ─────────────────────────────────────────────────────────────

print("Downloading ShareGPT dataset from HuggingFace...")

try:
    # Try HuggingFace datasets library first
    dataset = load_dataset(
        "anon8231489123/ShareGPT_Vicuna_unfiltered",
        data_files="ShareGPT_V3_unfiltered_cleaned_split.json",
        split="train"
    )
    conversations = list(dataset)
    print(f"Loaded {len(conversations)} conversations via HuggingFace")

except Exception as e:
    print(f"HuggingFace load failed: {e}")
    print("Trying direct download...")

    import requests
    url = (
        "https://huggingface.co/datasets/anon8231489123/"
        "ShareGPT_Vicuna_unfiltered/resolve/main/"
        "ShareGPT_V3_unfiltered_cleaned_split.json"
    )
    response = requests.get(url, timeout=120)
    if response.status_code == 200:
        conversations = response.json()
        print(f"Loaded {len(conversations)} conversations via direct download")
    else:
        raise RuntimeError(f"Download failed with status {response.status_code}")

# Preview one conversation
print("\nExample conversation structure:")
print(f"Keys: {conversations[0].keys()}")
print(f"First turn: {conversations[0]['conversations'][0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ShareGPT_V3_unfiltered_cleaned_split.jso(…):   0%|          | 0.00/673M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 94145 conversations via HuggingFace

Example conversation structure:
Keys: dict_keys(['id', 'conversations'])
First turn: {'from': 'human', 'markdown': None, 'text': None, 'value': "Summarize the main ideas of Jeff Walker's Product Launch Formula into bullet points as it pertains to a growth marketing agency implementing these strategies and tactics for their clients..."}


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Preprocess Dataset
# Extract clean prompt-response pairs from conversations
# ─────────────────────────────────────────────────────────────

from transformers import AutoTokenizer

print("Loading tokenizer for OPT-1.3B...")
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")
print("Tokenizer loaded!")

MAX_INPUT_LEN  = 512   # keep prompts short so T4 doesn't OOM
MAX_OUTPUT_LEN = 512   # limit outputs for faster experiments
MAX_SEQ_LEN    = MAX_INPUT_LEN + MAX_OUTPUT_LEN
BLOCK_SIZE     = 16    # vLLM default KV block size (tokens per block)
NUM_SAMPLES    = 500   # number of requests to run through vLLM
                       # increase to 1000+ if you have time

def classify_task(text: str) -> str:
    """Rule-based task classifier based on prompt content."""
    text_lower = text.lower()
    if len(text) < 100:
        return "short_query"
    elif any(w in text_lower for w in
             ["translate", "translation", "french", "german", "spanish", "chinese"]):
        return "translation"
    elif any(w in text_lower for w in
             ["```", "code", "function", "program", "python", "java", "algorithm"]):
        return "code"
    elif any(w in text_lower for w in
             ["explain", "what is", "how does", "describe", "define"]):
        return "explanation"
    else:
        return "general_chat"

def preprocess_conversations(conversations: list,
                              n: int = NUM_SAMPLES) -> List[Dict]:
    """
    Extract valid prompt-response pairs from ShareGPT conversations.
    Returns list of dicts with prompt text and metadata.
    """
    samples = []

    for conv in conversations:
        if len(samples) >= n:
            break

        turns = conv.get('conversations', [])

        # Find first human→gpt pair
        prompt_text   = None
        response_text = None

        for i, turn in enumerate(turns):
            if turn.get('from') == 'human' and prompt_text is None:
                prompt_text = turn.get('value', '').strip()
            elif turn.get('from') == 'gpt' and prompt_text is not None:
                response_text = turn.get('value', '').strip()
                break

        if not prompt_text or not response_text:
            continue

        # Tokenize to get real token counts
        prompt_tokens   = tokenizer.encode(prompt_text, truncation=True,
                                           max_length=MAX_INPUT_LEN)
        response_tokens = tokenizer.encode(response_text, truncation=True,
                                           max_length=MAX_OUTPUT_LEN)

        input_len  = len(prompt_tokens)
        output_len = len(response_tokens)

        # Skip very short samples — not informative
        if input_len < 10 or output_len < 10:
            continue

        # Assign random sampling parameters to simulate varied requests
        temperature = float(np.random.choice(
            [0.3, 0.5, 0.7, 0.8, 1.0, 1.2],
            p=[0.05, 0.10, 0.25, 0.35, 0.20, 0.05]
        ))
        max_tokens = int(np.random.choice(
            [128, 256, 512],
            p=[0.20, 0.50, 0.30]
        ))

        samples.append({
            'prompt'           : tokenizer.decode(prompt_tokens),
            'prompt_token_ids' : prompt_tokens,
            'input_length'     : input_len,
            'output_length'    : output_len,   # ground truth — used after generation
            'temperature'      : temperature,
            'max_tokens'       : max_tokens,
            'task_type'        : classify_task(prompt_text),
            'question_marks'   : prompt_text.count('?'),
            'instruction_words': sum(1 for w in
                ["explain", "describe", "write", "list", "analyze", "compare", "summarize"]
                if w in prompt_text.lower()),
        })

    print(f"Preprocessed {len(samples)} valid samples from {len(conversations)} conversations")
    return samples

np.random.seed(42)
samples = preprocess_conversations(conversations, n=NUM_SAMPLES)

# Show distribution
df_preview = pd.DataFrame(samples)
print(f"\nInput length  — mean: {df_preview['input_length'].mean():.1f}, "
      f"std: {df_preview['input_length'].std():.1f}")
print(f"Output length — mean: {df_preview['output_length'].mean():.1f}, "
      f"std: {df_preview['output_length'].std():.1f}")
print(f"\nTask type distribution:")
print(df_preview['task_type'].value_counts())

Loading tokenizer for OPT-1.3B...


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Tokenizer loaded!
Preprocessed 500 valid samples from 94145 conversations

Input length  — mean: 136.6, std: 181.2
Output length — mean: 295.2, std: 158.0

Task type distribution:
task_type
general_chat    186
short_query     178
code             78
explanation      48
translation      10
Name: count, dtype: int64


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Patch vLLM to Log KV Block Usage
#
# vLLM doesn't expose KV block counts in its public API.
# We monkey-patch the scheduler's _schedule() method to
# record how many physical GPU blocks are used per request.
# ─────────────────────────────────────────────────────────────

from vllm import LLM, SamplingParams
from vllm.engine.llm_engine import LLMEngine
from vllm.core.scheduler import Scheduler

# Dictionary to store block usage per request_id
kv_block_log: Dict[str, int] = {}

# Save reference to original schedule method
_original_schedule = Scheduler.schedule

def _patched_schedule(self):
    """
    Patched version of vLLM's Scheduler.schedule().
    After the original scheduling step, we check completed
    sequences and log how many blocks they used.
    """
    # Call the original schedule method
    scheduler_output = _original_schedule(self)

    # Log block usage for sequences that just finished
    # block_manager tracks physical block allocations per sequence
    for seq_group in scheduler_output.scheduled_seq_groups:
        for seq in seq_group.seq_group.get_seqs():
            if seq.is_finished():
                request_id = seq_group.seq_group.request_id
                # Count allocated physical blocks for this sequence
                try:
                    blocks = self.block_manager.get_block_table(seq)
                    kv_block_log[request_id] = len(blocks)
                except Exception:
                    # Fallback: estimate from sequence length
                    total_len = seq.get_len()
                    kv_block_log[request_id] = int(np.ceil(total_len / BLOCK_SIZE))

    return scheduler_output

# Apply the monkey patch
Scheduler.schedule = _patched_schedule
print("vLLM scheduler patched successfully!")

ModuleNotFoundError: No module named 'vllm.core'